<a href="https://colab.research.google.com/github/dordor7/SkillReal-Home-Assignment/blob/main/SkillReal_ML_Engineer_Home_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installation of LoHi-Weld
We will clone the repository from GitHub and install the package using pip. We also need to install `rdkit` which is a common dependency for molecular/chemical projects like LoHi.

In [2]:
# Install rdkit (often required for chemical informatics tools like LoHi)
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 37.0 MB/s  0:00:00


In [3]:
import os
import subprocess

# Clone the repository if it doesn't already exist
repo_dir = "/content/LoHi-Weld"
if not os.path.exists(repo_dir):
    print("Cloning LoHi-Weld repository...")
    subprocess.run(["git", "clone", "https://github.com/SylvioBlock/LoHi-Weld.git", repo_dir], check=True)
else:
    print("Repository already cloned.")

# Change working directory to the project folder
os.chdir(repo_dir)
print(f"Current working directory: {os.getcwd()}")

Cloning LoHi-Weld repository...
Current working directory: /content/LoHi-Weld


In [4]:
# Clean requirements.txt of absolute/local conda paths AND remove strict version pinning to prevent compilation issues
import re

with open('requirements.txt', 'r') as f:
    lines = f.readlines()

cleaned_lines = []
for line in lines:
    line_str = line.strip()
    # Skip empty lines or comments
    if not line_str or line_str.startswith('#'):
        continue
    # Skip absolute local paths
    if line_str.startswith('/') or 'file://' in line_str:
        continue

    # Strip out strict '==' version pins so pip fetches modern, pre-compiled wheels compatible with python 3.13
    package_name = re.split(r'==|>=|<=|>|<', line_str)[0].strip()
    if package_name:
        cleaned_lines.append(package_name + '\n')

with open('requirements_cleaned.txt', 'w') as f:
    f.writelines(cleaned_lines)

!pip install -r requirements_cleaned.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 36.4 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.7 MB/s  0:00:00
  Attempting uninstall: tbb
    Found existing installation: tbb 2022.3.1
    Uninstalling tbb-2022.3.1:
      Successfully uninstalled tbb-2022.3.1
  Attempting uninstall: mkl
    Found existing installation: mkl 2025.3.1
    Uninstalling mkl-2025.3.1:
      Successfully uninstalled mkl-2025.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mkl-service]


In [5]:
import gdown
import zipfile
import os

# Google Drive file ID from the provided URL
file_id = '1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK'
download_url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'weld-dataset.zip'

print("Downloading the dataset from Google Drive...")
gdown.download(download_url, output_zip, quiet=False)

print("Extracting the dataset...")
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall('.')

# Optional: Clean up the downloaded ZIP file
if os.path.exists(output_zip):
    os.remove(output_zip)

print("Dataset download and extraction complete!")

Downloading...
From (original): https://drive.google.com/uc?id=1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK
From (redirected): https://drive.google.com/uc?id=1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK&confirm=t&uuid=1adb66f6-490f-477f-ba54-16bb57d22aa6
To: /content/LoHi-Weld/weld-dataset.zip
100%|██████████| 631M/631M [00:05<00:00, 126MB/s]


Extracting the dataset...
Dataset download and extraction complete!


In [6]:
# Remove annotations for classes other than 'pore' and 'deposit'

import json
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")

for subset in ("low_resolution_welds", "high_resolution_welds"):
    folder = root / subset

    for path in folder.glob("*.yolo"):
        lines = path.read_text(encoding="utf-8").splitlines()
        kept = [
            line for line in lines
            if line.strip() and line.split()[0] in {"0", "1"}
        ]
        path.write_text("".join(line + "\n" for line in kept), encoding="utf-8")

    for path in folder.glob("*.json"):
        annotation = json.loads(path.read_text(encoding="utf-8"))
        annotation["shapes"] = [
            shape for shape in annotation["shapes"]
            if shape["label"] in {"pore", "deposit"}
        ]
        path.write_text(json.dumps(annotation, indent=2) + "\n", encoding="utf-8")

    print(f"Filtered {subset}")

Filtered low_resolution_welds
Filtered high_resolution_welds


In [7]:
import os
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

for subset in ("low_resolution_welds", "high_resolution_welds"):
    folder = root / subset
    if not folder.exists():
        print(f"Folder {folder} does not exist, skipping.")
        continue

    deleted_count = 0
    # Gather all image stems to inspect their annotations
    image_files = [p for p in folder.iterdir() if p.suffix.lower() in image_extensions]

    print(f"--- Cleaning unannotated images in {subset} (starting with {len(image_files)} images) ---")

    for img_path in image_files:
        stem = img_path.stem
        yolo_path = folder / f"{stem}.yolo"
        json_path = folder / f"{stem}.json"

        has_annotations = False

        # Check YOLO file
        if yolo_path.exists():
            lines = yolo_path.read_text(encoding="utf-8").splitlines()
            # If there's at least one non-empty line after filtering, it has annotations
            if any(line.strip() for line in lines):
                has_annotations = True

        # Check JSON file (fallback/double-check)
        if not has_annotations and json_path.exists():
            try:
                import json
                annotation = json.loads(json_path.read_text(encoding="utf-8"))
                if annotation.get("shapes"):
                    has_annotations = True
            except Exception:
                pass

        # If no active annotations, delete the image and its helper files
        if not has_annotations:
            try:
                img_path.unlink()
                if yolo_path.exists():
                    yolo_path.unlink()
                if json_path.exists():
                    json_path.unlink()
                deleted_count += 1
            except Exception as e:
                print(f"Error deleting {stem}: {e}")

    remaining_images = [p for p in folder.iterdir() if p.suffix.lower() in image_extensions]
    print(f"Deleted {deleted_count} unannotated image groups.")
    print(f"Remaining images in {subset}: {len(remaining_images)}\n")

--- Cleaning unannotated images in low_resolution_welds (starting with 2000 images) ---
Deleted 994 unannotated image groups.
Remaining images in low_resolution_welds: 1006

--- Cleaning unannotated images in high_resolution_welds (starting with 1022 images) ---
Deleted 194 unannotated image groups.
Remaining images in high_resolution_welds: 828



In [9]:
import pandas as pd
from pathlib import Path
from collections import Counter

root = Path("/content/LoHi-Weld/weld-dataset")

# Dictionary to hold the final counts
counts = {
    "low": Counter(),
    "high": Counter()
}

# Map internal directory structures to our key names
subsets = {
    "low": root / "low_resolution_welds",
    "high": root / "high_resolution_welds"
}

# Iterate and count annotations from .yolo files
for key, folder in subsets.items():
    if not folder.exists():
        continue
    for yolo_file in folder.glob("*.yolo"):
        try:
            lines = yolo_file.read_text(encoding="utf-8").splitlines()
            for line in lines:
                parts = line.strip().split()
                if parts:
                    cls = parts[0]
                    if cls in {"0", "1"}:
                        counts[key][cls] += 1
        except Exception:
            pass

# Map classes to descriptive names (0 = pore, 1 = deposit)
class_names = {"0": "Pore (Class 0)", "1": "Deposit (Class 1)"}

# Create a structured DataFrame for display
data = {
    "Annotation Class": [class_names["0"], class_names["1"]],
    "Low Resolution Count": [counts["low"]["0"], counts["low"]["1"]],
    "High Resolution Count": [counts["high"]["0"], counts["high"]["1"]],
    "Total": [counts["low"]["0"] + counts["high"]["0"], counts["low"]["1"] + counts["high"]["1"]]
}

summary_annotations_df = pd.DataFrame(data)
display(summary_annotations_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

Annotation Class,Low Resolution Count,High Resolution Count,Total
Pore (Class 0),3427,523,3950
Deposit (Class 1),1141,1794,2935
